# Olive Fly Detection using Neural Network

This notebook implements an olive fly detection system using a lightweight neural network.
It includes both training and classification pipelines for detecting olive flies in olive fruit images.

## 1. Import Required Libraries

In [67]:
import os
import glob
import cv2
import numpy as np
from skimage.measure import label
from sklearn.neural_network import MLPClassifier

## 2. Image Processing Functions

### 2.1 Extract Foreground Mask
This function extracts the olive fruit (foreground) from the background using threshold and morphological operations.

In [68]:
def extract_foreground_mask(img, kernel_size=9):
    """Extract foreground (olive fruit) from the image."""
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.uint8)
    _, img_bw = cv2.threshold(img_gray, -1, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    kernel = np.ones((kernel_size, kernel_size))
    img_bw_cleaned = cv2.morphologyEx(img_bw, cv2.MORPH_CLOSE, kernel)
    
    labels = label(img_bw_cleaned)
    flat_labels = labels.flat
    flat_bw = img_bw_cleaned.flat
    
    if len(flat_bw) == 0 or np.sum(flat_bw) == 0:
        return np.zeros_like(img_gray)
        
    label_of_largest_region = np.argmax(np.bincount(flat_labels, weights=flat_bw))
    return (labels == label_of_largest_region).astype(np.uint8)

### 2.2 Compute Image Features
Extract three key features from each image:
- **Area**: Size of the foreground object
- **Aspect Ratio**: Width-to-height ratio of the bounding rectangle
- **Mean Saturation**: Average color saturation (useful for detecting the fly's coloring)

In [69]:
def compute_features(image):
    """Compute feature vector from image."""
    mask = extract_foreground_mask(image)
    area = float(np.sum(mask))
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        aspect_ratio = float(w) / float(h) if h > 0 else 1.0
    else:
        aspect_ratio = 1.0
        
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    saturation_channel = hsv[:, :, 1]
    mean_saturation = float(cv2.mean(saturation_channel, mask=mask)[0])
    
    return np.array([area, aspect_ratio, mean_saturation])

## 3. Training Phase

Load training data from the `olive_fly/` and `not_olive_fly/` directories and train the neural network classifier.

In [70]:
# Prepare training data
X_list, y_list = [], []

print("Reading training data...")

# Load negative samples (no olive fly)
for f in (glob.glob("not_olive_fly/*.JPG") + glob.glob("not_olive_fly/*.jpg")):
    img = cv2.imread(f)
    if img is not None:
        X_list.append(compute_features(img))
        y_list.append(0)
            
# Load positive samples (contains olive fly)
for f in (glob.glob("olive_fly/*.JPG") + glob.glob("olive_fly/*.jpg")):
    img = cv2.imread(f)
    if img is not None:
        X_list.append(compute_features(img))
        y_list.append(1)

print(f"Loaded {len(X_list)} training samples")
print(f"Positive samples (olive fly): {sum(y_list)}")
print(f"Negative samples (no olive fly): {len(y_list) - sum(y_list)}")

Reading training data...
Loaded 4700 training samples
Positive samples (olive fly): 630
Negative samples (no olive fly): 4070


### Feature Standardization and Model Training

In [71]:
# Convert to numpy arrays
X, y = np.array(X_list), np.array(y_list)

# Feature Standardization Scaling
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_std[X_std == 0] = 1e-8  # Avoid division by zero
X_scaled = (X - X_mean) / X_std

print("Feature statistics:")
print(f"X_mean: {X_mean}")
print(f"X_std: {X_std}")

# Train Neural Network: 3 inputs -> 4 hidden neurons -> 1 output
mlp = MLPClassifier(hidden_layer_sizes=(4,), activation='relu', max_iter=5000, random_state=42)
mlp.fit(X_scaled, y)

print("\nNeural Network Training Complete!")

Feature statistics:
X_mean: [3.87745872e+03 1.11746390e+00 1.49344926e+02]
X_std: [4.80163557e+03 5.52435110e-01 5.18820731e+01]

Neural Network Training Complete!


### Extract and Display Model Parameters

Copy these parameters for use in the classification phase:

In [72]:
print(f"X_MEAN = np.array({list(X_mean)})")
print(f"X_STD  = np.array({list(X_std)})")
print(f"W1 = np.array({mlp.coefs_[0].tolist()})")
print(f"B1 = np.array({mlp.intercepts_[0].tolist()})")
print(f"W2 = np.array({mlp.coefs_[1].tolist()})")
print(f"B2 = np.array({mlp.intercepts_[1].tolist()})")

# Store the parameters for later use
MODEL_PARAMS = {
    'X_MEAN': X_mean,
    'X_STD': X_std,
    'W1': mlp.coefs_[0],
    'B1': mlp.intercepts_[0],
    'W2': mlp.coefs_[1],
    'B2': mlp.intercepts_[1]
}

X_MEAN = np.array([np.float64(3877.458723404255), np.float64(1.1174638977603828), np.float64(149.3449262126882)])
X_STD  = np.array([np.float64(4801.635565207151), np.float64(0.5524351102376486), np.float64(51.88207312731143)])
W1 = np.array([[-1.4787926909184586, 0.9596893415034591, 1.093172432294036, -1.7454627691021827], [-0.26691265456305974, -1.811491088397156, 0.4203204015223413, 0.38288509439103313], [-0.6746690403539322, 0.17595712582269765, -1.535109766765984, 1.3510070511987022]])
B1 = np.array([0.6876098416420647, -0.8068542754680046, 0.7945428630582831, -1.1543043806619804])
W2 = np.array([[-0.814867770461975], [-0.9349578475965044], [-0.8518958607413155], [-1.6780657705355944]])
B2 = np.array([0.24297322103842567])


## 4. Classification Phase

### 4.1 Neural Network Parameters
These are the pre-trained weights from the training phase (or can be replaced with your own trained parameters):

In [73]:
# Use the trained model parameters from above, or paste your own:
X_MEAN = MODEL_PARAMS['X_MEAN']
X_STD = MODEL_PARAMS['X_STD']

W1 = MODEL_PARAMS['W1']
B1 = MODEL_PARAMS['B1']

W2 = MODEL_PARAMS['W2']
B2 = MODEL_PARAMS['B2']

print("Model parameters loaded successfully!")

Model parameters loaded successfully!


### 4.2 Detection Function
Implements the forward pass of the neural network for classification:

In [74]:
def detect_olive_fly(image) -> bool:
    """
    Classifies olive flies using an ultra-lightweight Neural Network forward pass.
    
    Args:
        image: Input image (BGR format from OpenCV)
    
    Returns:
        bool: True if olive fly detected, False otherwise
    """
    try:
        # 1. Extract and standardize physical attributes
        raw_features = compute_features(image)
        scaled_features = (raw_features - X_MEAN) / X_STD
        
        # 2. Hidden Layer Propagation: Z1 = A1*W1 + B1
        z1 = np.dot(scaled_features, W1) + B1
        a2 = np.maximum(0, z1)  # ReLU non-linear activation function
        
        # 3. Output Layer Propagation: Z2 = A2*W2 + B2
        z2 = np.dot(a2, W2) + B2
        
        # Sigmoid activation function mapping to classification boundaries
        probability = 1 / (1 + np.exp(-np.clip(z2[0], -500, 500)))
        
        return bool(probability >= 0.5)
        
    except Exception as e:
        print(f"Error in detection: {e}")
        return False

### 4.3 Evaluation on Test Data

In [75]:
def evaluate_model(test_directory, verbose=False):
    """
    Evaluate the model on test data.
    
    Args:
        test_directory: Path to directory containing test images
        verbose: Print detailed results for each image
    """
    TP, TN, FP, FN = 0, 0, 0, 0
    
    for filename in glob.glob(test_directory + "/**/*.JPG", recursive=True) + \
                    glob.glob(test_directory + "/**/*.jpg", recursive=True):
        img = cv2.imread(filename)
        if img is None:
            continue
            
        # Determine ground truth from directory name
        if "not_olive_fly" in filename:
            olive_fly = False
        elif "olive_fly" in filename:
            olive_fly = True
        else:
            print(f"Skipping {filename} - not labeled")
            continue

        detection_result = detect_olive_fly(img)

        # Update confusion matrix
        if olive_fly and detection_result:
            TP += 1
        elif olive_fly and not detection_result:
            FN += 1
        elif not olive_fly and detection_result:
            FP += 1
        else:
            TN += 1
            
        if verbose:
            result_str = "contains an olive fly" if detection_result else "does not contain an olive fly"
            print(f"{filename} {result_str}")
    
    print(f"\nConfusion Matrix:")
    print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")
    
    if (TP + FP) > 0 and (TP + FN) > 0:
        precision = TP / (TP + FP)
        recall = TP / (TP + FN)
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f"\nMetrics:")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1_score:.4f}")
    
    return TP, TN, FP, FN

### 4.4 Run Evaluation
Uncomment and modify the path below to evaluate on your test dataset:

In [76]:

print("Evaluating on training data...\n")
evaluate_model(".", verbose=False)


Evaluating on training data...


Confusion Matrix:
TP: 26, TN: 4036, FP: 34, FN: 604

Metrics:
Precision: 0.4333
Recall: 0.0413
F1-Score: 0.0754


(26, 4036, 34, 604)

## 5. Test Single Image
Use this section to test the detection on a single image:

In [ ]:
test_image_path = "olive_fly\IMG_0597 1 referencia.JPG"


if os.path.exists(test_image_path):
    test_img = cv2.imread(test_image_path)
    if test_img is not None:
        result = detect_olive_fly(test_img)
        print(f"Image: {test_image_path}")
        print(f"Detection Result: {'Olive Fly Detected' if result else 'No Olive Fly'}")
    else:
        print(f"Could not read image: {test_image_path}")
else:
    print(f"File not found: {test_image_path}")

Image: not_olive_fly/castellar_2_1 201 referencia.JPG
Detection Result: No Olive Fly
